In [1]:
from pathlib import Path

In [2]:
from delta import configure_spark_with_delta_pip

In [3]:
from pyspark.sql import SparkSession

In [6]:
builder = (
    SparkSession.builder
    .appName("LakehouseLab")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

In [7]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 13:34:27 WARN Utils: Your hostname, muon, resolves to a loopback address: 127.0.1.1; using 192.168.178.47 instead (on interface wlp3s0)
26/10/11 13:34:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ygor/Documents/lakehouse-lab/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ygor/.ivy2.5.2/cache
The jars for the packages stored in: /home/ygor/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-91eb5c46-6cac-4f92-b13b-63bf45705708;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.13/

In [8]:
print("Spark version:", spark.version)

Spark version: 4.0.0


In [10]:
table_path = str(
    Path.home() / "lakehouse-lab" / "data" / "sales_delta"
)

In [11]:
df = spark.createDataFrame(
    [
        (1, "A", 100.0),
        (2, "B", 150.0),
        (3, "A", 200.0),
    ],
    ["id", "category", "amount"],
)

In [12]:
df.write.format("delta").mode("overwrite").save(table_path)

26/10/11 13:38:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [17]:
spark.read.format("delta").load(table_path) \
    .groupBy("category") \
    .sum("amount") \
    .show()

+--------+-----------+
|category|sum(amount)|
+--------+-----------+
|       A|      300.0|
|       B|      150.0|
+--------+-----------+



In [18]:
import mlflow

In [19]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [20]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("lakehouse-lab")

2026/10/11 14:14:47 INFO mlflow.tracking.fluent: Experiment with name 'lakehouse-lab' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1791717287928, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791717287928, lifecycle_stage='active', name='lakehouse-lab', tags={}, trace_location=None, workspace='default'>

In [21]:
X, y = load_diabetes(return_X_y=True)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
with mlflow.start_run():
    model = RandomForestRegressor(
        n_estimators=100, random_state=42
    )
    model.fit(X_train, y_train)

    mse = mean_squared_error(y_test, model.predict(X_test))

    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("test_mse", mse)

print("Test MSE:", mse)

🏃 View run upset-skink-711 at: http://127.0.0.1:5000/#/experiments/1/runs/5a0db63ef2364f0589cb6fe9bf7cb3e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Test MSE: 2952.0105887640448


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 37638)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/home/ygor/Documents/lakehouse-lab/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/home/ygor/Documents/lakehouse-lab/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/ygor/Documents/lakeh